In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/akshataudaganve/sentiment-claude/final_train_dataset.csv
/kaggle/input/datasets/akshataudaganve/sentiment-claude/final_test_dataset.csv


In [2]:
import numpy as np
import pandas as pd
import torch
import json
import glob
import os
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

train_df = pd.read_csv('/kaggle/input/datasets/akshataudaganve/sentiment-claude/final_train_dataset.csv').dropna(subset=['text', 'label'])
test_df  = pd.read_csv('/kaggle/input/datasets/akshataudaganve/sentiment-claude/final_test_dataset.csv').dropna(subset=['text', 'label'])

train_df['strata'] = train_df['lang'] + '_' + train_df['label'].astype(str)
inner_train, inner_val = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['strata'])
inner_train = inner_train.drop(columns=['strata'])
inner_val = inner_val.drop(columns=['strata'])

train_ds = Dataset.from_pandas(inner_train[['text', 'label']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(inner_val[['text', 'label']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

def find_latest_checkpoint(output_dir):
    checkpoints = glob.glob(f"{output_dir}/checkpoint-*")
    if not checkpoints:
        return None
    return sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1]

def train_and_evaluate(model_name, output_dir):
    final_save_path = f"{output_dir}/final_best_model"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(examples):
        return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

    train_tok = train_ds.map(tokenize_fn, batched=True)
    val_tok   = val_ds.map(tokenize_fn, batched=True)
    test_tok  = test_ds.map(tokenize_fn, batched=True)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=6,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=2,
    )

    # ---------------- CASE 1: already fully trained — skip training, just reload + evaluate ----------------
    if os.path.exists(final_save_path):
        print(f"\n{'='*60}\n{model_name} — ALREADY COMPLETE, skipping training, loading saved model\n{'='*60}")
        model = AutoModelForSequenceClassification.from_pretrained(final_save_path)
        tokenizer = AutoTokenizer.from_pretrained(final_save_path)

        trainer = Trainer(
            model=model, args=args,
            train_dataset=train_tok, eval_dataset=val_tok,
            compute_metrics=compute_metrics,
            processing_class=tokenizer,
        )

    else:
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

        trainer = Trainer(
            model=model, args=args,
            train_dataset=train_tok, eval_dataset=val_tok,
            compute_metrics=compute_metrics,
            processing_class=tokenizer,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        # ---------------- CASE 2: partial checkpoint exists — resume from it ----------------
        latest_ckpt = find_latest_checkpoint(output_dir)
        if latest_ckpt:
            print(f"\n{'='*60}\n{model_name} — RESUMING from checkpoint: {latest_ckpt}\n{'='*60}")
            trainer.train(resume_from_checkpoint=latest_ckpt)
        # ---------------- CASE 3: nothing exists — train fresh ----------------
        else:
            print(f"\n{'='*60}\n{model_name} — starting FRESH training (no checkpoint found)\n{'='*60}")
            trainer.train()

        trainer.save_model(final_save_path)
        tokenizer.save_pretrained(final_save_path)
        print(f"Saved final model + tokenizer to: {final_save_path}")

    # ---------------- EVALUATION (runs every time, cheap even if training was skipped) ----------------
    test_results = trainer.predict(test_tok)
    preds = np.argmax(test_results.predictions, axis=1)
    labels = test_df['label'].values

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    cm = confusion_matrix(labels, preds)

    print(f"\n{'='*60}\n{model_name} — OVERALL TEST RESULTS\n{'='*60}")
    print(f"Accuracy:  {acc:.4f}\nPrecision: {precision:.4f}\nRecall:    {recall:.4f}\nF1:        {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    per_lang = {}
    tdf = test_df.copy()
    tdf['pred'] = preds
    print(f"\nPer-language results for {model_name}:")
    for lang in sorted(tdf['lang'].unique()):
        sub = tdf[tdf['lang'] == lang]
        lang_acc = accuracy_score(sub['label'], sub['pred'])
        lp, lr, lf1, _ = precision_recall_fscore_support(sub['label'], sub['pred'], average='binary', zero_division=0)
        per_lang[lang] = {'accuracy': lang_acc, 'precision': lp, 'recall': lr, 'f1': lf1, 'n': len(sub)}
        print(f"  {lang}: acc={lang_acc:.4f}  prec={lp:.4f}  rec={lr:.4f}  f1={lf1:.4f}  (n={len(sub)})")

    return {'model_name': model_name, 'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1,
            'confusion_matrix': cm, 'per_language': per_lang, 'final_model_path': final_save_path}

muril_results = train_and_evaluate("google/muril-base-cased", "./muril_output")
xlmr_results  = train_and_evaluate("xlm-roberta-base", "./xlmr_output")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

The OrderedVocab you are attempting to save contains holes for indices [202, 437, 1046, 1057, 1118, 1135, 1150, 1162, 1318, 1445, 1473, 1610, 1626, 1775, 3517, 3643, 4513, 5830, 7834, 12787, 13244, 19712, 25184, 27726, 28024, 31739, 65274], your vocabulary could be corrupted!


Map:   0%|          | 0/159936 [00:00<?, ? examples/s]

... (54 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model + tokenizer to: ./xlmr_output/final_best_model


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



xlm-roberta-base — OVERALL TEST RESULTS
Accuracy:  0.8275
Precision: 0.8236
Recall:    0.8831
F1:        0.8523
Confusion Matrix:
[[6110 1977]
 [1222 9231]]

Per-language results for xlm-roberta-base:
  en: acc=0.8386  prec=0.8202  rec=0.8758  f1=0.8471  (n=5000)
  hi: acc=0.8212  prec=0.8014  rec=0.8608  f1=0.8300  (n=5000)
  kn: acc=0.8209  prec=0.8402  rec=0.9235  f1=0.8799  (n=497)
  ta: acc=0.8501  prec=0.8838  rec=0.9411  f1=0.9116  (n=3043)
  te: acc=0.8094  prec=0.7879  rec=0.8496  f1=0.8176  (n=5000)


In [3]:
print(f"\n{'='*60}\nFINAL COMPARISON — MuRIL vs XLM-R\n{'='*60}")
comparison = pd.DataFrame([
    {'Model': 'MuRIL', 'Accuracy': muril_results['accuracy'], 'Precision': muril_results['precision'],
     'Recall': muril_results['recall'], 'F1': muril_results['f1']},
    {'Model': 'XLM-R', 'Accuracy': xlmr_results['accuracy'], 'Precision': xlmr_results['precision'],
     'Recall': xlmr_results['recall'], 'F1': xlmr_results['f1']},
])
print(comparison.to_string(index=False))

winner = 'MuRIL' if muril_results['f1'] > xlmr_results['f1'] else 'XLM-R'
winner_path = muril_results['final_model_path'] if winner == 'MuRIL' else xlmr_results['final_model_path']
print(f"\nBetter overall model (by F1): {winner}")
print(f"Winning model saved at: {winner_path}")

comparison.to_csv('model_comparison_overall.csv', index=False)

lang_comparison = []
for lang in sorted(muril_results['per_language'].keys()):
    lang_comparison.append({
        'Language': lang,
        'MuRIL F1': muril_results['per_language'][lang]['f1'],
        'XLM-R F1': xlmr_results['per_language'][lang]['f1'],
        'Better': 'MuRIL' if muril_results['per_language'][lang]['f1'] > xlmr_results['per_language'][lang]['f1'] else 'XLM-R',
    })
lang_comparison_df = pd.DataFrame(lang_comparison)
print(lang_comparison_df.to_string(index=False))
lang_comparison_df.to_csv('model_comparison_per_language.csv', index=False)


FINAL COMPARISON — MuRIL vs XLM-R
Model  Accuracy  Precision   Recall       F1
MuRIL  0.823193   0.820570 0.878504 0.848549
XLM-R  0.827454   0.823608 0.883096 0.852315

Better overall model (by F1): XLM-R
Winning model saved at: ./xlmr_output/final_best_model
Language  MuRIL F1  XLM-R F1 Better
      en  0.835663  0.847130  XLM-R
      hi  0.818407  0.830038  XLM-R
      kn  0.886515  0.879892  MuRIL
      ta  0.915866  0.911559  MuRIL
      te  0.818925  0.817608  MuRIL


In [4]:
SENTIMENT_MODEL_PATH = winner_path
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_PATH)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH)
sentiment_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
sentiment_model.to(device)

LABEL_MAP = {0: "negative", 1: "positive"}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
def add_sentiment_to_tweet(tweet_record: dict) -> dict:
    """
    Input: a dict already containing at least {"tweet_id", "text", "lang", "is_disaster", ...}
           — whatever fields the earlier pipeline steps (language detection, disaster
           classification) already attached.
    Output: the SAME dict, with sentiment fields added on top. Nothing removed.
    """
    text = tweet_record["text"]

    inputs = sentiment_tokenizer(text, truncation=True, padding="max_length",
                                  max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = sentiment_model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze().tolist()

    pred_class = int(torch.argmax(logits, dim=1).item())

    # add new fields, keep everything already in the record (lang, is_disaster, etc.)
    tweet_record["sentiment_label"] = LABEL_MAP[pred_class]
    tweet_record["sentiment_confidence"] = round(probs[pred_class], 4)
    tweet_record["sentiment_probabilities"] = {LABEL_MAP[i]: round(p, 4) for i, p in enumerate(probs)}
    tweet_record["sentiment_negativity"] = round(probs[0], 4)   # feeds into Severity formula
    tweet_record["sentiment_model"] = SENTIMENT_MODEL_PATH
    tweet_record["sentiment_predicted_at"] = datetime.now(timezone.utc).isoformat()

    return tweet_record

def add_sentiment_to_tweets_batch(tweet_records: list, batch_size: int = 32) -> list:
    """Processes a list of tweet JSON records efficiently, all at once per batch."""
    texts = [r["text"] for r in tweet_records]
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = sentiment_tokenizer(batch, truncation=True, padding=True,
                                      max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = sentiment_model(**inputs).logits
            probs = torch.softmax(logits, dim=1).cpu().tolist()
        all_probs.extend(probs)

    for record, probs in zip(tweet_records, all_probs):
        pred_class = int(np.argmax(probs))
        record["sentiment_label"] = LABEL_MAP[pred_class]
        record["sentiment_confidence"] = round(probs[pred_class], 4)
        record["sentiment_probabilities"] = {LABEL_MAP[i]: round(p, 4) for i, p in enumerate(probs)}
        record["sentiment_negativity"] = round(probs[0], 4)
        record["sentiment_model"] = SENTIMENT_MODEL_PATH
        record["sentiment_predicted_at"] = datetime.now(timezone.utc).isoformat()

    return tweet_records




incoming_tweets = [
    {"tweet_id": "tw_00981", "text": "flood in my area need help urgently", "lang": "en", "is_disaster": True},
    {"tweet_id": "tw_00982", "text": "ನಮ್ಮ ಏರಿಯಾದಲ್ಲಿ ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು", "lang": "kn", "is_disaster": True},
    {"tweet_id": "tw_00983", "text": "watching a movie tonight", "lang": "en", "is_disaster": False},
]

updated_tweets = add_sentiment_to_tweets_batch(incoming_tweets)

# save as JSON file — this is what gets passed to the next pipeline step (clustering)
with open("tweets_with_sentiment.json", "w", encoding="utf-8") as f:
    json.dump(updated_tweets, f, ensure_ascii=False, indent=2)

print(json.dumps(updated_tweets, ensure_ascii=False, indent=2))

[
  {
    "tweet_id": "tw_00981",
    "text": "flood in my area need help urgently",
    "lang": "en",
    "is_disaster": true,
    "sentiment_label": "negative",
    "sentiment_confidence": 0.9943,
    "sentiment_probabilities": {
      "negative": 0.9943,
      "positive": 0.0057
    },
    "sentiment_negativity": 0.9943,
    "sentiment_model": "./xlmr_output/final_best_model",
    "sentiment_predicted_at": "2026-09-02T00:59:30.426972+00:00"
  },
  {
    "tweet_id": "tw_00982",
    "text": "ನಮ್ಮ ಏರಿಯಾದಲ್ಲಿ ಪ್ರವಾಹ ಬಂದಿದೆ ಸಹಾಯ ಬೇಕು",
    "lang": "kn",
    "is_disaster": true,
    "sentiment_label": "negative",
    "sentiment_confidence": 0.9697,
    "sentiment_probabilities": {
      "negative": 0.9697,
      "positive": 0.0303
    },
    "sentiment_negativity": 0.9697,
    "sentiment_model": "./xlmr_output/final_best_model",
    "sentiment_predicted_at": "2026-09-02T00:59:30.427005+00:00"
  },
  {
    "tweet_id": "tw_00983",
    "text": "watching a movie tonight",
    "lang": "en",
  

In [6]:
import shutil

# zips the winning model folder into a single downloadable file
shutil.make_archive('xlmr_sentiment_model_final', 'zip', winner_path)

print("Saved as: xlmr_sentiment_model_final.zip")
print("Find it in the Output panel on the right side of the Kaggle editor, and click download.")

Saved as: xlmr_sentiment_model_final.zip
Find it in the Output panel on the right side of the Kaggle editor, and click download.
